# 06 - Grouping and Aggregations

Aggregations answer questions about groups of rows, such as revenue and order count by region.

## Learning objectives

By the end of this notebook, you will be able to:

- explain how grouping changes dataset grain;
- calculate and name several measures;
- distinguish row counts from non-null value counts; and
- recognise equivalent DataFrame and Spark SQL queries.

## Prerequisite recap

Earlier transformations worked one row at a time. `groupBy` brings rows with the same key together, which commonly requires a shuffle.

In [ ]:
from pyspark.sql import functions as F

retail_sales = spark.createDataFrame(
    [
        (1001, 'C001', 'North', 'Stationery', 37.00),
        (1002, 'C002', 'West', 'Furniture', 750.00),
        (1003, 'C001', 'North', 'Stationery', 36.00),
        (1004, 'C003', 'North', 'Stationery', 92.50),
        (1005, 'C004', 'South', 'Electronics', 170.00),
        (1006, 'C002', 'West', 'Stationery', 48.00),
        (1007, 'C003', 'North', 'Furniture', 750.00),
        (1008, 'C004', 'South', 'Stationery', None),
    ],
    'order_id INT, customer_id STRING, region STRING, category STRING, order_value DOUBLE',
)
retail_sales.show()

## Group rows, then calculate

The source grain is one row per order. `groupBy('region')` followed by an aggregation produces one row per region.

In [ ]:
retail_sales.groupBy('region').count().show()
retail_sales.groupBy('region').sum('order_value').show()

## Calculate several measures

Use `agg` to calculate several measures together and `alias` to give each output a business-friendly name.

`count('*')` counts rows, including rows with null values. `count('order_value')` counts only rows where `order_value` is not null.

In [ ]:
region_summary = retail_sales.groupBy('region').agg(
    F.sum('order_value').alias('total_revenue'),
    F.count('*').alias('order_count'),
    F.count('order_value').alias('priced_order_count'),
    F.countDistinct('customer_id').alias('distinct_customers'),
    F.avg('order_value').alias('average_order_value'),
    F.min('order_value').alias('smallest_order_value'),
    F.max('order_value').alias('largest_order_value'),
).orderBy(F.col('total_revenue').desc())

region_summary.show()

## Group by more than one key

Passing several columns creates one result row for each unique combination.

In [ ]:
region_category_summary = retail_sales.groupBy('region', 'category').agg(
    F.sum('order_value').alias('total_revenue'),
    F.count('*').alias('order_count'),
)
region_category_summary.orderBy('region', 'category').show()

## A Spark SQL bridge

The DataFrame API and Spark SQL use the same Spark engine. A temporary view is available only in the current Spark session.

In [ ]:
retail_sales.createOrReplaceTempView('retail_sales')

spark.sql('''
    SELECT
        region,
        SUM(order_value) AS total_revenue,
        COUNT(*) AS order_count
    FROM retail_sales
    GROUP BY region
    ORDER BY total_revenue DESC
''').show()

## Your turn

Use this separate support-ticket dataset.

In [ ]:
support_tickets = spark.createDataFrame(
    [
        ('T001', 'Billing', 4.0),
        ('T002', 'Billing', None),
        ('T003', 'Billing', 1.0),
        ('T004', 'Technical', 8.0),
        ('T005', 'Technical', 2.0),
        ('T006', 'Shipping', 3.0),
        ('T007', 'Shipping', 5.0),
    ],
    'ticket_id STRING, team STRING, resolution_hours DOUBLE',
)
support_tickets.show()

Create `team_summary` with one row per support team. Calculate `ticket_count`, `resolved_ticket_count`, `fastest_resolution_hours`, and `slowest_resolution_hours`. Sort by ticket count descending, then team ascending.

In [ ]:
# Write your solution here.

### Expected result

There are three team rows. Billing has three tickets but only two resolved tickets; Technical and Shipping each have two resolved tickets.

### Solution - reveal after attempting

In [ ]:
team_summary = support_tickets.groupBy('team').agg(
    F.count('*').alias('ticket_count'),
    F.count('resolution_hours').alias('resolved_ticket_count'),
    F.min('resolution_hours').alias('fastest_resolution_hours'),
    F.max('resolution_hours').alias('slowest_resolution_hours'),
).orderBy(F.col('ticket_count').desc(), F.col('team').asc())

team_summary.show()

## Key takeaway

Always state the grain of the input and output, name aggregate columns clearly, and remember that most aggregate functions ignore null values.

**Next:** combine related DataFrames horizontally and vertically.